# Charlie Eden

4/10/2026

In [1]:
# Setting working dir
import os
from pathlib import Path
p = os.getcwd() + "/../../"
os.chdir(p)
parent_dir = os.getcwd()

In [35]:
import numpy as np

config = {
    "years": np.arange(2019, 2025),
    "common_period": np.arange(2004, 2022),
    "imd_folder": f"{parent_dir}/../monsoon-benchmark_data/imd_rainfall_data/4p0",  # Ground truth rainfall data (4x4 degrees)
    "thresh_file": f"{parent_dir}/../monsoon-benchmark_data/imd_onset_threshold/mwset4x4.nc4",  # Threshold for the onset of the monsoon (4x4 degrees)
    "thres_file": f"{parent_dir}/../monsoon-benchmark_data/imd_onset_threshold/mwset4x4.nc4",  # Alternate naming convention
    "shpfile_path": f"{parent_dir}/../monsoon-benchmark_data/ind_map_shpfile/india_shapefile.shp",  # Shapefile of India
    "output_dir": f"{parent_dir}/examples/paper_figures/outputs",  # Directory to save data files,
    "mok": True,
    "day_bins_30": [(1, 5), (6, 10), (11, 15), (16, 20), (21, 25), (26, 30)],
    "day_bins_15": [(1, 5), (6, 10), (11, 15)],
    "mem_num": 51,
    "date_filter_year": 2024,
    "file_pattern": "{}.nc",
    "data_dir": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0",
}

# Paths to 4p0 model forecast data (.nc)
model_paths = {
    "IFS": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/IFS_S2S",  # IFS model
    "AIFS": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/AIFS",  # AIFS model
    "FuXi": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/FuXi",  # FuXi mdoel
    "Graphcast": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/GraphCast",  # Graphcast model
    "GenCast": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/GenCast",  # GenCast model
    "FuXi-S2S": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/FuXi_S2S",  # FuXi_S2S model
    "NGCM": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/NeuralGCM",  # NGCM model
}

prob_model_paths = {
        "FuXi S2S": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/FuXi_S2S",  # FuXi_S2S model
        "NGCM": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/NeuralGCM",  # NGCM model
        "IFS": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/IFS_S2S",  # AIFS model
        "GenCast": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/GenCast",  # GenCast model
    }

det_model_paths = {
        "AIFS": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/AIFS",
        "FuXi": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/FuXi",
        "Graphcast": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/GraphCast",  # Graphcast model
    }

In [36]:
from monsoonbench.metrics import (
    ProbabilisticOnsetMetrics,
    ClimatologyOnsetMetrics,
    DeterministicOnsetMetrics
)
from monsoonbench.visualization.compare_models import calculate_reliability_metrics
import xarray as xr

c = ClimatologyOnsetMetrics()
p = ProbabilisticOnsetMetrics()
d = DeterministicOnsetMetrics()

skills_15 = {}
skills_30 = {}
brier_15 = {}
brier_30 = {}
auc_15 = {}
auc_30 = {}
reliability_15 = {}

thresh_ds = xr.open_dataset(config["thresh_file"])
thresh_slice = thresh_ds["MWmean"]
clim_onset = c.compute_climatological_onset_dataset(
            config["imd_folder"], thresh_slice, years=None, mok=config["mok"]
        )

Processing 124 years: [1901, 1902, 1903, 1904, 1905, 1906, 1907, 1908, 1909, 1910, 1911, 1912, 1913, 1914, 1915, 1916, 1917, 1918, 1919, 1920, 1921, 1922, 1923, 1924, 1925, 1926, 1927, 1928, 1929, 1930, 1931, 1932, 1933, 1934, 1935, 1936, 1937, 1938, 1939, 1940, 1941, 1942, 1943, 1944, 1945, 1946, 1947, 1948, 1949, 1950, 1951, 1952, 1953, 1954, 1955, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Processing year 1901...
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/1901.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd)

### Fig 2 -- Probabilistic Scores

In [5]:
# 15 day
for model_name in [
    "IFS",
    "GenCast",
    "FuXi-S2S",
    "NGCM",
    ]:
    print("\n")
    print(f"Processing {model_name}")
    print("="*50)
    forecast_df = p.multi_year_forecast_obs_pairs(
            range(2019, 2024),
            model_forecast_dir=model_paths[model_name],
            imd_folder=config["imd_folder"],
            thres_file=config["thresh_file"],
            mem_num=51,
            max_forecast_day=15,
            day_bins=config["day_bins_15"],
            date_filter_year=config["date_filter_year"],
        )
    
    climatology_obs_df = c.multi_year_climatological_forecast_obs_pairs(
            clim_onset,
            config["years"],
            config["day_bins_15"],
            config["mem_num"],
            model_paths[model_name],
            date_filter_year=config["date_filter_year"],
            file_pattern=config["file_pattern"],
            max_forecast_day=15,
            mok=config["mok"],
        )
    
    brier_forecast = p.calculate_brier_score(forecast_df)
    rps_forecast = p.calculate_rps(forecast_df)
    auc_forecast = p.calculate_auc(forecast_df)

    brier_climatology = c.calculate_brier_score_climatology(climatology_obs_df)
    rps_climatology = p.calculate_rps(climatology_obs_df)
    auc_climatology = c.calculate_auc_climatology(climatology_obs_df)


    skill_results = p.calculate_skill_scores(
            brier_forecast, rps_forecast, brier_climatology, rps_climatology
        )
    
    reliability_metrics = calculate_reliability_metrics(forecast_df)

    skills_15[model_name] = skill_results
    reliability_15[model_name] = reliability_metrics
    auc_15[model_name] = auc_forecast
    brier_15[model_name] = brier_forecast
        



Processing IFS
Processing years: range(2019, 2024)


Using 4-degree CMZ polygon coordinates

Processing year 2019
Loading S2S model data...
Loading IMD rainfall data...
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/2019.nc
Renamed dimensions: {'TIME': 'time'}
Detecting observed onset...
Using MOK date (June 2nd) (2019-06-02) as start date for onset detection
Found onset in 100 out of 100 grid points
Computing onset for all ensemble members...
Processing 26 init times x 10 unique locations x 11 members...
Unique lat-lon pairs: [(np.float64(76.0), np.float64(20.0)), (np.float64(80.0), np.float64(20.0)), (np.float64(84.0), np.float64(20.0)), (np.float64(72.0), np.float64(24.0)), (np.float64(76.0), np.float64(24.0)), (np.float64(80.0), np.float64(24.0)), (np.float64(84.0), np.float64(24.0)), (np.float64(72.0), np.float64(28.0)), (np.float64(76.0), np.float64(28.0)), (np.float64(80.0), np.float64(28.0))]
Using MOK (June 2nd filter) for onset detection


Traceback (most recent call last):
  File "/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py", line 1003, in multi_year_forecast_obs_pairs
    ProbabilisticOnsetMetrics.get_forecast_probabilistic_twice_weekly_2(
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        year,
        ^^^^^
    ...<3 lines>...
        file_pattern,
        ^^^^^^^^^^^^^
    )
    ^
  File "/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py", line 120, in get_forecast_probabilistic_twice_weekly_2
    raise FileNotFoundError(f"File not found: {file_path}")
FileNotFoundError: File not found: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/rainfall_4p0/FuXi_S2S/2022.nc
Traceback (most recent call last):
  File "/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py", line 1003, in multi_year_forecast_

Generated 390 climatological forecast-observation pairs
Unique lat-lon pairs processed: 10
Total bins per forecast: 5
Probability range: 0.000 - 1.000
Observed onset rate: 0.200
Non-zero probabilities: 228
Unique locations in output: 10

Distribution across bins:
                      predicted_prob        observed_onset  \
                               count   mean           mean   
bin_label                                                    
After day 15                      78  0.677          0.731   
Before initialization             78  0.108          0.000   
Days 1-5                          78  0.065          0.077   
Days 11-15                        78  0.082          0.103   
Days 6-10                         78  0.067          0.090   

                      n_contributing_years total_members_with_onset  
                                      mean                     mean  
bin_label                                                            
After day 15                 

In [6]:
# 30 day
for model_name in [
    "IFS",
    "GenCast",
    "FuXi-S2S",
    "NGCM",
    ]:
    print("\n")
    print(f"Processing {model_name}")
    print("="*50)
    forecast_df = p.multi_year_forecast_obs_pairs(
            range(2019, 2024),
            model_forecast_dir=model_paths[model_name],
            imd_folder=config["imd_folder"],
            thres_file=config["thresh_file"],
            mem_num=51,
            max_forecast_day=30,
            day_bins=config["day_bins_30"],
            date_filter_year=config["date_filter_year"],
        )
    
    climatology_obs_df = c.multi_year_climatological_forecast_obs_pairs(
            clim_onset,
            config["years"],
            config["day_bins_30"],
            config["mem_num"],
            model_paths[model_name],
            date_filter_year=config["date_filter_year"],
            file_pattern=config["file_pattern"],
            max_forecast_day=30,
            mok=config["mok"],
        )
    
    brier_forecast = p.calculate_brier_score(forecast_df)
    rps_forecast = p.calculate_rps(forecast_df)
    auc_forecast = p.calculate_auc(forecast_df)

    brier_climatology = c.calculate_brier_score_climatology(climatology_obs_df)
    rps_climatology = p.calculate_rps(climatology_obs_df)
    auc_climatology = c.calculate_auc_climatology(climatology_obs_df)


    skill_results = p.calculate_skill_scores(
            brier_forecast, rps_forecast, brier_climatology, rps_climatology
        )
    
    skills_30[model_name] = skill_results  
    auc_30[model_name] = auc_forecast
    brier_30[model_name] = brier_forecast      



Processing IFS
Processing years: range(2019, 2024)
Using 4-degree CMZ polygon coordinates

Processing year 2019
Loading S2S model data...
Loading IMD rainfall data...
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/2019.nc
Renamed dimensions: {'TIME': 'time'}
Detecting observed onset...
Using MOK date (June 2nd) (2019-06-02) as start date for onset detection
Found onset in 100 out of 100 grid points
Computing onset for all ensemble members...
Processing 26 init times x 10 unique locations x 11 members...
Unique lat-lon pairs: [(np.float64(76.0), np.float64(20.0)), (np.float64(80.0), np.float64(20.0)), (np.float64(84.0), np.float64(20.0)), (np.float64(72.0), np.float64(24.0)), (np.float64(76.0), np.float64(24.0)), (np.float64(80.0), np.float64(24.0)), (np.float64(84.0), np.float64(24.0)), (np.float64(72.0), np.float64(28.0)), (np.float64(76.0), np.float64(28.0)), (np.float64(80.0), np.float64(28.

Traceback (most recent call last):
  File "/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py", line 1003, in multi_year_forecast_obs_pairs
    ProbabilisticOnsetMetrics.get_forecast_probabilistic_twice_weekly_2(
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        year,
        ^^^^^
    ...<3 lines>...
        file_pattern,
        ^^^^^^^^^^^^^
    )
    ^
  File "/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py", line 120, in get_forecast_probabilistic_twice_weekly_2
    raise FileNotFoundError(f"File not found: {file_path}")
FileNotFoundError: File not found: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/rainfall_4p0/FuXi_S2S/2022.nc
Traceback (most recent call last):
  File "/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py", line 1003, in multi_year_forecast_

Generated 624 climatological forecast-observation pairs
Unique lat-lon pairs processed: 10
Total bins per forecast: 8
Probability range: 0.000 - 1.000
Observed onset rate: 0.125
Non-zero probabilities: 402
Unique locations in output: 10

Distribution across bins:
                      predicted_prob        observed_onset  \
                               count   mean           mean   
bin_label                                                    
After day 30                      78  0.417          0.462   
Before initialization             78  0.108          0.000   
Days 1-5                          78  0.065          0.077   
Days 11-15                        78  0.082          0.103   
Days 16-20                        78  0.088          0.077   
Days 21-25                        78  0.086          0.115   
Days 26-30                        78  0.086          0.077   
Days 6-10                         78  0.067          0.090   

                      n_contributing_years total_memb

In [9]:
import pandas as pd
rows = []
for model_name in [
    "IFS",
    "GenCast",
    "FuXi-S2S",
    "NGCM",
    ]:
    row = {}
    row["model_label"] = model_name.lower()
    row["horizon"] = 15

    row["fair_brier_skill_d1_5"] = skills_15[model_name]["bin_fair_brier_skill_scores"]["Days 1-5"]
    row["fair_brier_skill_d6_10"] = skills_15[model_name]["bin_fair_brier_skill_scores"]["Days 6-10"]
    row["fair_brier_skill_d11_15"] = skills_15[model_name]["bin_fair_brier_skill_scores"]["Days 11-15"]
    row["fair_brier_skill_d16_20"] = None
    row["fair_brier_skill_d21_25"] = None
    row["fair_brier_skill_d26_30"] = None

    row["fair_brier_d1_5"] = brier_15[model_name]["bin_fair_brier_scores"]["Days 1-5"]
    row["fair_brier_d6_10"] = brier_15[model_name]["bin_fair_brier_scores"]["Days 6-10"]
    row["fair_brier_d11_15"] = brier_15[model_name]["bin_fair_brier_scores"]["Days 11-15"]
    row["fair_brier_d16_20"] = None
    row["fair_brier_d21_25"] = None
    row["fair_brier_d26_30"] = None

    row["auc_d1_5"] = auc_15[model_name]["bin_auc_scores"]["Days 1-5"]
    row["auc_d6_10"] = auc_15[model_name]["bin_auc_scores"]["Days 6-10"]
    row["auc_d11_15"] = auc_15[model_name]["bin_auc_scores"]["Days 11-15"]
    row["auc_d16_20"] = None
    row["auc_d21_25"] = None
    row["auc_d26_30"] = None
    row["auc_later"] = auc_15[model_name]["bin_auc_scores"]["After day 15"]

    rows.append(row)


    row = {}
    row["model_label"] = model_name.lower()
    row["horizon"] = 30

    row["fair_brier_skill_d1_5"] = skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 1-5"]
    row["fair_brier_skill_d6_10"] = skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 6-10"]
    row["fair_brier_skill_d11_15"] = skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 11-15"]
    row["fair_brier_skill_d16_20"] = skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 16-20"]
    row["fair_brier_skill_d21_25"] = skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 21-25"]
    row["fair_brier_skill_d26_30"] = skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 26-30"]

    row["fair_brier_d1_5"] = brier_30[model_name]["bin_fair_brier_scores"]["Days 1-5"]
    row["fair_brier_d6_10"] = brier_30[model_name]["bin_fair_brier_scores"]["Days 6-10"]
    row["fair_brier_d11_15"] = brier_30[model_name]["bin_fair_brier_scores"]["Days 11-15"]
    row["fair_brier_d16_20"] = brier_30[model_name]["bin_fair_brier_scores"]["Days 16-20"]
    row["fair_brier_d21_25"] = brier_30[model_name]["bin_fair_brier_scores"]["Days 21-25"]
    row["fair_brier_d26_30"] = brier_30[model_name]["bin_fair_brier_scores"]["Days 26-30"]

    row["auc_d1_5"] = auc_30[model_name]["bin_auc_scores"]["Days 1-5"]
    row["auc_d6_10"] = auc_30[model_name]["bin_auc_scores"]["Days 6-10"]
    row["auc_d11_15"] = auc_30[model_name]["bin_auc_scores"]["Days 11-15"]
    row["auc_d16_20"] = auc_30[model_name]["bin_auc_scores"]["Days 16-20"]
    row["auc_d21_25"] = auc_30[model_name]["bin_auc_scores"]["Days 21-25"]
    row["auc_d26_30"] = auc_30[model_name]["bin_auc_scores"]["Days 26-30"]
    row["auc_later"] = auc_30[model_name]["bin_auc_scores"]["After day 30"]

    rows.append(row)

metrics_df = pd.DataFrame(rows)
metrics_df


,model_label,horizon,fair_brier_skill_d1_5,fair_brier_skill_d6_10,fair_brier_skill_d11_15,fair_brier_skill_d16_20,fair_brier_skill_d21_25,fair_brier_skill_d26_30,fair_brier_d1_5,fair_brier_d6_10,...,fair_brier_d16_20,fair_brier_d21_25,fair_brier_d26_30,auc_d1_5,auc_d6_10,auc_d11_15,auc_d16_20,auc_d21_25,auc_d26_30,auc_later
0,ifs,15,0.397846,0.147049,0.046467,NaN,NaN,NaN,0.048682,0.068558,...,NaN,NaN,NaN,0.922698,0.851670,0.771822,NaN,NaN,NaN,0.927304
1,ifs,30,0.397846,0.147049,0.046467,0.049100,-0.001361,0.013516,0.048682,0.068558,...,0.079129,0.087224,0.086659,0.922698,0.851670,0.771822,0.766231,0.694732,0.721722,0.895437
2,gencast,15,0.332756,0.065693,0.052427,NaN,NaN,NaN,0.052584,0.075229,...,NaN,NaN,NaN,0.929903,0.833178,0.816743,NaN,NaN,NaN,0.929464
3,gencast,30,0.332756,0.065693,0.052427,-0.052475,-0.044604,-0.027344,0.052584,0.075229,...,0.084940,0.090560,0.088514,0.929903,0.833178,0.816743,0.774316,0.728810,0.731957,0.900925
4,fuxi-s2s,15,0.208563,0.069433,-0.155639,NaN,NaN,NaN,0.061938,0.067932,...,NaN,NaN,NaN,0.808628,0.871501,0.661179,NaN,NaN,NaN,0.872859
5,fuxi-s2s,30,0.208563,0.069433,-0.155639,-0.081604,-0.272543,-0.018465,0.061938,0.067932,...,0.094841,0.110850,0.078090,0.808628,0.871501,0.661179,0.748228,0.651966,0.774356,0.900549
6,ngcm,15,0.334993,0.124527,0.055870,NaN,NaN,NaN,0.052407,0.070492,...,NaN,NaN,NaN,0.947398,0.875928,0.824400,NaN,NaN,NaN,0.938979
7,ngcm,30,0.334993,0.124527,0.055870,-0.005738,-0.047950,-0.092803,0.052407,0.070492,...,0.081168,0.090850,0.094154,0.947398,0.875928,0.824400,0.812459,0.725694,0.682558,0.908256


In [12]:
metrics_df.to_csv(f"{config["output_dir"]}/fig2_data_recreation.csv")

In [33]:
reliability_15

{'IFS':   Bin_Range  N_Forecasts  Mean_Forecast_Prob  Observed_Frequency  Frequency  \
 0   0.0-0.1         1674               0.012               0.038      0.619   
 1   0.1-0.2          161               0.182               0.205      0.060   
 2   0.2-0.3          106               0.273               0.311      0.039   
 3   0.3-0.4           92               0.364               0.261      0.034   
 4   0.4-0.5           73               0.455               0.397      0.027   
 5   0.5-0.6           51               0.545               0.412      0.019   
 6   0.6-0.7           42               0.636               0.524      0.016   
 7   0.7-0.8           52               0.727               0.635      0.019   
 8   0.8-0.9           46               0.818               0.630      0.017   
 9   0.9-1.0          407               0.986               0.953      0.151   
 
    Error_Bar  bin_center  
 0      0.005        0.05  
 1      0.032        0.15  
 2      0.045        0.25  

### Fig 3 -- Binned Avg. CMZ Spatial Metrics

In [13]:
# Reading in Matlab files

from scipy.io import loadmat

def load_mat_to_dict(file_path:str,
                    vars_of_interest: list = None):
    data = loadmat(file_path)
    if vars_of_interest:
        ret_dict = {var: data[var] for var in vars_of_interest}
        return ret_dict
    return data

In [14]:
load_mat_to_dict("/Users/charlieeden/Downloads/5day_forecastwindow_cmz_2019_2024.mat")

{'__header__': b'MATLAB 5.0 MAT-file, Platform: MACA64, Created on: Sat Sep 27 12:35:10 2025',
 '__version__': '1.0',
 '__globals__': [],
 'far_cmz': array([[ 5.00552672,  5.14806667,  6.14221866,  1.61055272,  8.92096044,
          4.60774198,  3.59489348,  5.85507257],
        [ 7.95982277,  8.78547899,  8.85290505,  4.26848899, 11.39773605,
          6.4457009 ,  6.9739043 ,  7.40127785],
        [10.5148626 ,  7.35800489, 12.77523487,  6.54565838, 13.09821372,
          4.49552206,  6.85812311,  4.00694431],
        [12.0407337 ,  6.28736664, 12.38162626, 13.88521342, 24.13860657,
          6.12166779, 11.62840659,  6.82676204],
        [14.46997348,  7.46808236, 17.49078923, 15.73044724, 27.4567295 ,
          6.41197107, 11.71757763, 10.33545782],
        [16.13182367, 10.74482817, 23.85337689, 20.23400135, 17.59355116,
          9.77286226, 18.68398268,  6.96542738]]),
 'mae_cmz': array([[ 4.52380952,  2.82559524,  2.352403  ,  1.51871693,  3.23525463,
          2.48729608,  2.4

In [21]:
from monsoonbench.utils.forecast_window_pipeline import (
    run_multi_model_window_analysis,
    get_default_four_degree_model_specs
)

model_specs = {
        "ifs": {
            "model_type": "probabilistic",
            "expected_ens": 11,
            "mem_num": 11,
            "date_filter_year": 2024,
            "path": model_paths["IFS"],
        },
        "aifs": {
            "model_type": "deterministic",
            "expected_ens": None,
            "mem_num": None,
            "date_filter_year": 2024,
            "path": model_paths["AIFS"],
        },
        "fuxi": {
            "model_type": "deterministic",
            "expected_ens": None,
            "mem_num": None,
            "date_filter_year": 2024,
            "path": model_paths["FuXi"],
        },
        "graphcast": {
            "model_type": "deterministic",
            "expected_ens": None,
            "mem_num": None,
            "date_filter_year": 2024,
            "path": model_paths["Graphcast"],
        },
        "gencast": {
            "model_type": "probabilistic",
            "expected_ens": 51,
            "mem_num": 51,
            "date_filter_year": 2024,
            "path": model_paths["GenCast"],
        },
        "fuxi-s2s": {
            "model_type": "probabilistic",
            "expected_ens": 51,
            "mem_num": 51,
            "date_filter_year": 2022,
            "path": model_paths["FuXi-S2S"],
        },
        "ngcm": {
            "model_type": "probabilistic",
            "expected_ens": 51,
            "mem_num": 51,
            "date_filter_year": 2024,
            "path": model_paths["NGCM"],
        },
    }

fig3_spatial_cmz_data = run_multi_model_window_analysis(base_config=config,
                                model_specs=model_specs,
                                )


Running model=ifs, years=[2019, 2020, 2021, 2022, 2023]
Computing climatological onset from 124 years: 1901-2024
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/1901.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1901-06-02) as start date for onset detection
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/1902.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1902-06-02) as start date for onset detection
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/1903.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1903-06-02) as start date for onset detection
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/im

In [62]:
for key, value in fig3_spatial_cmz_data["matrices"].items():
    value.to_csv(
        f"{config["output_dir"]}/fig3_data_recreation_{key}.csv"
    )
    print(f"CSV for {key} saved")

CSV for mae_cmz saved
CSV for far_cmz saved
CSV for mr_cmz saved
CSV for std_er saved


In [61]:
common_skills_15 = {}
common_skills_30 = {}
common_brier_15 = {}
common_brier_30 = {}
common_auc_15 = {}
common_auc_30 = {}
common_reliability_15 = {}

In [31]:
# 15 day
for model_name in [
    "IFS",
    "GenCast",
    "FuXi-S2S",
    "NGCM",
    ]:
    print("\n")
    print(f"Processing {model_name}")
    print("="*50)
    forecast_df = p.multi_year_forecast_obs_pairs(
            config["common_period"],
            model_forecast_dir=model_paths[model_name],
            imd_folder=config["imd_folder"],
            thres_file=config["thresh_file"],
            mem_num=51,
            max_forecast_day=15,
            day_bins=config["day_bins_15"],
            date_filter_year=2022,
        )
    
    climatology_obs_df = c.multi_year_climatological_forecast_obs_pairs(
            clim_onset,
            config["common_period"],
            config["day_bins_15"],
            config["mem_num"],
            model_paths[model_name],
            date_filter_year=2022,
            file_pattern=config["file_pattern"],
            max_forecast_day=15,
            mok=config["mok"],
        )
    
    brier_forecast = p.calculate_brier_score(forecast_df)
    rps_forecast = p.calculate_rps(forecast_df)
    auc_forecast = p.calculate_auc(forecast_df)

    brier_climatology = c.calculate_brier_score_climatology(climatology_obs_df)
    rps_climatology = p.calculate_rps(climatology_obs_df)
    auc_climatology = c.calculate_auc_climatology(climatology_obs_df)


    skill_results = p.calculate_skill_scores(
            brier_forecast, rps_forecast, brier_climatology, rps_climatology
        )
    
    reliability_metrics = calculate_reliability_metrics(forecast_df)

    common_skills_15[model_name] = skill_results
    common_reliability_15[model_name] = reliability_metrics
    common_auc_15[model_name] = auc_forecast
    common_brier_15[model_name] = brier_forecast

# 30 day
for model_name in [
    "IFS",
    "GenCast",
    "FuXi-S2S",
    "NGCM",
    ]:
    print("\n")
    print(f"Processing {model_name}")
    print("="*50)
    forecast_df = p.multi_year_forecast_obs_pairs(
            config['common_period'],
            model_forecast_dir=model_paths[model_name],
            imd_folder=config["imd_folder"],
            thres_file=config["thresh_file"],
            mem_num=51,
            max_forecast_day=30,
            day_bins=config["day_bins_30"],
            date_filter_year=2022,
        )
    
    climatology_obs_df = c.multi_year_climatological_forecast_obs_pairs(
            clim_onset,
            config["common_period"],
            config["day_bins_30"],
            config["mem_num"],
            model_paths[model_name],
            date_filter_year=2022,
            file_pattern=config["file_pattern"],
            max_forecast_day=30,
            mok=config["mok"],
        )
    
    brier_forecast = p.calculate_brier_score(forecast_df)
    rps_forecast = p.calculate_rps(forecast_df)
    auc_forecast = p.calculate_auc(forecast_df)

    brier_climatology = c.calculate_brier_score_climatology(climatology_obs_df)
    rps_climatology = p.calculate_rps(climatology_obs_df)
    auc_climatology = c.calculate_auc_climatology(climatology_obs_df)


    skill_results = p.calculate_skill_scores(
            brier_forecast, rps_forecast, brier_climatology, rps_climatology
        )
    
    common_skills_30[model_name] = skill_results  
    common_auc_30[model_name] = auc_forecast
    common_brier_30[model_name] = brier_forecast      
        



Processing IFS
Processing years: [2004 2005 2006 2007 2008 2009 2010 2011 2012 2013 2014 2015 2016 2017
 2018 2019 2020 2021]
Using 4-degree CMZ polygon coordinates

Processing year 2004
Loading S2S model data...
Loading IMD rainfall data...
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/2004.nc
Renamed dimensions: {'TIME': 'time'}
Detecting observed onset...
Using MOK date (June 2nd) (2004-06-02) as start date for onset detection
Found onset in 100 out of 100 grid points
Computing onset for all ensemble members...
Processing 13 init times x 10 unique locations x 11 members...
Unique lat-lon pairs: [(np.float64(76.0), np.float64(20.0)), (np.float64(80.0), np.float64(20.0)), (np.float64(84.0), np.float64(20.0)), (np.float64(72.0), np.float64(24.0)), (np.float64(76.0), np.float64(24.0)), (np.float64(80.0), np.float64(24.0)), (np.float64(84.0), np.float64(24.0)), (np.float64(72.0), np.float64(28.0

Traceback (most recent call last):
  File "/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py", line 1003, in multi_year_forecast_obs_pairs
    ProbabilisticOnsetMetrics.get_forecast_probabilistic_twice_weekly_2(
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        year,
        ^^^^^
    ...<3 lines>...
        file_pattern,
        ^^^^^^^^^^^^^
    )
    ^
  File "/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py", line 120, in get_forecast_probabilistic_twice_weekly_2
    raise FileNotFoundError(f"File not found: {file_path}")
FileNotFoundError: File not found: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/rainfall_4p0/GenCast/2004.nc
Traceback (most recent call last):
  File "/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py", line 1003, in multi_year_forecast_o

Found onset in 100 out of 100 grid points
Computing onset for all ensemble members...
Processing 13 init times x 10 unique locations x 51 members...
Unique lat-lon pairs: [(np.float64(76.0), np.float64(20.0)), (np.float64(80.0), np.float64(20.0)), (np.float64(84.0), np.float64(20.0)), (np.float64(72.0), np.float64(24.0)), (np.float64(76.0), np.float64(24.0)), (np.float64(80.0), np.float64(24.0)), (np.float64(84.0), np.float64(24.0)), (np.float64(72.0), np.float64(28.0)), (np.float64(76.0), np.float64(28.0)), (np.float64(80.0), np.float64(28.0))]
Using MOK (June 2nd filter) for onset detection
Processing init time 1/13: 2019-05-02
Processing init time 6/13: 2019-06-06
Processing init time 11/13: 2019-07-11

Processing Summary:
Total potential forecasts: 6630
Skipped (no observed onset): 0
Skipped (initialized after observed onset): 2652
Valid forecasts processed: 3978
Generated 3978 member-forecast combinations
Found onset in 868 cases
Onset rate: 0.218
✓ All init_time-lat-lon-member co

Traceback (most recent call last):
  File "/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py", line 1003, in multi_year_forecast_obs_pairs
    ProbabilisticOnsetMetrics.get_forecast_probabilistic_twice_weekly_2(
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        year,
        ^^^^^
    ...<3 lines>...
        file_pattern,
        ^^^^^^^^^^^^^
    )
    ^
  File "/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py", line 120, in get_forecast_probabilistic_twice_weekly_2
    raise FileNotFoundError(f"File not found: {file_path}")
FileNotFoundError: File not found: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/rainfall_4p0/GenCast/2004.nc
Traceback (most recent call last):
  File "/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py", line 1003, in multi_year_forecast_o

Processing init time 6/13: 2019-06-06
Processing init time 11/13: 2019-07-11

Processing Summary:
Total potential forecasts: 6630
Skipped (no observed onset): 0
Skipped (initialized after observed onset): 2652
Valid forecasts processed: 3978
Generated 3978 member-forecast combinations
Found onset in 1869 cases
Onset rate: 0.470
✓ All init_time-lat-lon-member combinations are unique
Found onset in 1869 member cases
Creating forecast-observation pairs...
Processing 78 forecast cases with day bins: [(1, 5), (6, 10), (11, 15), (16, 20), (21, 25), (26, 30)]
Including 'after day 30' bin for members without onset in forecast window
Generated 546 forecast-observation pairs
Total bins per forecast: 7
Probability range: 0.000 - 1.000
Observed onset rate: 0.143
Non-zero probabilities: 318

Distribution across bins:
             predicted_prob        observed_onset
                      count   mean           mean
bin_label                                        
After day 30             78  0.530

In [32]:
rows = []
for model_name in [
    "IFS",
    "GenCast",
    "FuXi-S2S",
    "NGCM",
    ]:
    row = {}
    row["model_label"] = model_name.lower()
    row["horizon"] = 15

    row["fair_brier_skill_d1_5"] = common_skills_15[model_name]["bin_fair_brier_skill_scores"]["Days 1-5"]
    row["fair_brier_skill_d6_10"] = common_skills_15[model_name]["bin_fair_brier_skill_scores"]["Days 6-10"]
    row["fair_brier_skill_d11_15"] = common_skills_15[model_name]["bin_fair_brier_skill_scores"]["Days 11-15"]
    row["fair_brier_skill_d16_20"] = None
    row["fair_brier_skill_d21_25"] = None
    row["fair_brier_skill_d26_30"] = None

    row["fair_brier_d1_5"] = common_brier_15[model_name]["bin_fair_brier_scores"]["Days 1-5"]
    row["fair_brier_d6_10"] = common_brier_15[model_name]["bin_fair_brier_scores"]["Days 6-10"]
    row["fair_brier_d11_15"] = common_brier_15[model_name]["bin_fair_brier_scores"]["Days 11-15"]
    row["fair_brier_d16_20"] = None
    row["fair_brier_d21_25"] = None
    row["fair_brier_d26_30"] = None

    row["auc_d1_5"] = common_auc_15[model_name]["bin_auc_scores"]["Days 1-5"]
    row["auc_d6_10"] = common_auc_15[model_name]["bin_auc_scores"]["Days 6-10"]
    row["auc_d11_15"] = common_auc_15[model_name]["bin_auc_scores"]["Days 11-15"]
    row["auc_d16_20"] = None
    row["auc_d21_25"] = None
    row["auc_d26_30"] = None
    row["auc_later"] = common_auc_15[model_name]["bin_auc_scores"]["After day 15"]

    rows.append(row)


    row = {}
    row["model_label"] = model_name.lower()
    row["horizon"] = 30

    row["fair_brier_skill_d1_5"] = common_skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 1-5"]
    row["fair_brier_skill_d6_10"] = common_skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 6-10"]
    row["fair_brier_skill_d11_15"] = common_skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 11-15"]
    row["fair_brier_skill_d16_20"] = common_skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 16-20"]
    row["fair_brier_skill_d21_25"] = common_skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 21-25"]
    row["fair_brier_skill_d26_30"] = common_skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 26-30"]

    row["fair_brier_d1_5"] = common_brier_30[model_name]["bin_fair_brier_scores"]["Days 1-5"]
    row["fair_brier_d6_10"] = common_brier_30[model_name]["bin_fair_brier_scores"]["Days 6-10"]
    row["fair_brier_d11_15"] = common_brier_30[model_name]["bin_fair_brier_scores"]["Days 11-15"]
    row["fair_brier_d16_20"] = common_brier_30[model_name]["bin_fair_brier_scores"]["Days 16-20"]
    row["fair_brier_d21_25"] = common_brier_30[model_name]["bin_fair_brier_scores"]["Days 21-25"]
    row["fair_brier_d26_30"] = common_brier_30[model_name]["bin_fair_brier_scores"]["Days 26-30"]

    row["auc_d1_5"] = common_auc_30[model_name]["bin_auc_scores"]["Days 1-5"]
    row["auc_d6_10"] = common_auc_30[model_name]["bin_auc_scores"]["Days 6-10"]
    row["auc_d11_15"] = common_auc_30[model_name]["bin_auc_scores"]["Days 11-15"]
    row["auc_d16_20"] = common_auc_30[model_name]["bin_auc_scores"]["Days 16-20"]
    row["auc_d21_25"] = common_auc_30[model_name]["bin_auc_scores"]["Days 21-25"]
    row["auc_d26_30"] = common_auc_30[model_name]["bin_auc_scores"]["Days 26-30"]
    row["auc_later"] = common_auc_30[model_name]["bin_auc_scores"]["After day 30"]

    rows.append(row)

common_metrics_df = pd.DataFrame(rows)
common_metrics_df


,model_label,horizon,fair_brier_skill_d1_5,fair_brier_skill_d6_10,fair_brier_skill_d11_15,fair_brier_skill_d16_20,fair_brier_skill_d21_25,fair_brier_skill_d26_30,fair_brier_d1_5,fair_brier_d6_10,...,fair_brier_d16_20,fair_brier_d21_25,fair_brier_d26_30,auc_d1_5,auc_d6_10,auc_d11_15,auc_d16_20,auc_d21_25,auc_d26_30,auc_later
0,ifs,15,0.244852,0.083944,0.112915,NaN,NaN,NaN,0.061957,0.062265,...,NaN,NaN,NaN,0.933867,0.826573,0.781372,NaN,NaN,NaN,0.933710
1,ifs,30,0.244852,0.083944,0.112915,0.027011,0.020428,0.026547,0.061957,0.062265,...,0.083511,0.077383,0.076728,0.933867,0.826573,0.781372,0.729617,0.682875,0.692172,0.903173
2,gencast,15,0.247295,-0.019477,0.057570,NaN,NaN,NaN,0.058907,0.074423,...,NaN,NaN,NaN,0.905018,0.809770,0.813391,NaN,NaN,NaN,0.910289
3,gencast,30,0.247295,-0.019477,0.057570,0.034114,-0.052971,-0.012585,0.058907,0.074423,...,0.084695,0.091723,0.077639,0.905018,0.809770,0.813391,0.806829,0.726904,0.720325,0.913482
4,fuxi-s2s,15,0.173327,0.032863,0.026668,NaN,NaN,NaN,0.068179,0.075226,...,NaN,NaN,NaN,0.893137,0.820004,0.795203,NaN,NaN,NaN,0.920527
5,fuxi-s2s,30,0.173327,0.032863,0.026668,0.028149,-0.023488,-0.005887,0.068179,0.075226,...,0.081889,0.083869,0.085724,0.893137,0.820004,0.795203,0.793127,0.765150,0.752192,0.897301
6,ngcm,15,0.181067,0.009371,0.068369,NaN,NaN,NaN,0.067191,0.067334,...,NaN,NaN,NaN,0.942232,0.837668,0.813054,NaN,NaN,NaN,0.938232
7,ngcm,30,0.181067,0.009371,0.068369,0.039376,0.002081,0.019421,0.067191,0.067334,...,0.082449,0.078832,0.077289,0.942232,0.837668,0.813054,0.778698,0.733676,0.753462,0.912899


In [64]:
common_metrics_df.to_csv(
    f"{config["output_dir"]}/fig3_common_period_data_recreation.csv"
)

### Fig 6 -- Probabilistic Metrics 4x4 lon/lat grid

In [43]:
brier_model_paths = {
        "FuXi-S2S": model_paths["FuXi-S2S"],
        "NGCM": model_paths["NGCM"],
        "IFS": model_paths["IFS"],
    }

In [44]:
from examples.paper_figures.fig6_through_12_utils import (
    calculate_gridwise_brier,
    calculate_gridwise_rps
)

forecast_dfs_15 = {}
forecast_dfs_30 = {}
for model_name, model_fp in brier_model_paths.items():
    print("=" * 80)
    print(f"Loading data from {model_name}")
    print("=" * 80)
    multi_year_df = p.multi_year_forecast_obs_pairs(
        range(2004, 2022),
        model_forecast_dir=model_fp,
        imd_folder=config["imd_folder"],
        thres_file=config["thresh_file"],
        mem_num=51 if model_name != "IFS" else 11,
        max_forecast_day=15,
        day_bins=config["day_bins_15"],
        date_filter_year=2024 if model_name != "IFS" else 2022,
    )
    forecast_dfs_15[model_name] = multi_year_df

    multi_year_df = p.multi_year_forecast_obs_pairs(
        range(2004, 2022),
        model_forecast_dir=model_fp,
        imd_folder=config["imd_folder"],
        thres_file=config["thresh_file"],
        mem_num=51 if model_name != "IFS" else 11,
        max_forecast_day=30,
        day_bins=config["day_bins_30"],
        date_filter_year=2024 if model_name != "IFS" else 2022,
    )
    forecast_dfs_30[model_name] = multi_year_df

brier_15_ds = {}
brier_30_ds = {}
rps_15_ds = {}
rps_30_ds = {}

for model, df in forecast_dfs_15.items():
    loop_grid_brier = calculate_gridwise_brier(df)
    brier_15_ds[model] = loop_grid_brier

for model, df in forecast_dfs_30.items():
    loop_grid_brier = calculate_gridwise_brier(df)
    brier_30_ds[model] = loop_grid_brier

for model, df in forecast_dfs_15.items():
    loop_grid_rps = calculate_gridwise_rps(df)
    rps_15_ds[model] = loop_grid_rps

for model, df in forecast_dfs_30.items():
    loop_grid_rps = calculate_gridwise_rps(df)
    rps_30_ds[model] = loop_grid_rps

brier_rps_graph_data_list = [brier_15_ds, brier_30_ds, rps_15_ds, rps_30_ds]

Loading data from FuXi-S2S
Processing years: range(2004, 2022)
Using 4-degree CMZ polygon coordinates

Processing year 2004
Loading S2S model data...
Loading IMD rainfall data...
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/2004.nc
Renamed dimensions: {'TIME': 'time'}
Detecting observed onset...
Using MOK date (June 2nd) (2004-06-02) as start date for onset detection
Found onset in 100 out of 100 grid points
Computing onset for all ensemble members...
Processing 13 init times x 10 unique locations x 51 members...
Unique lat-lon pairs: [(np.float64(76.0), np.float64(20.0)), (np.float64(80.0), np.float64(20.0)), (np.float64(84.0), np.float64(20.0)), (np.float64(72.0), np.float64(24.0)), (np.float64(76.0), np.float64(24.0)), (np.float64(80.0), np.float64(24.0)), (np.float64(84.0), np.float64(24.0)), (np.float64(72.0), np.float64(28.0)), (np.float64(76.0), np.float64(28.0)), (np.float64(80.0), np.f

/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py:1118: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  forecast_obs_df["squared_diff"] = squared_diffs
/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py:1119: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  forecast_obs_df["fair_brier_component"] = fair_brier_components
/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilis

Brier Score: 0.0906
Fair Brier Score: 0.0899
Brier Score: 0.0886
Fair Brier Score: 0.0881
Brier Score: 0.0870
Fair Brier Score: 0.0865
Brier Score: 0.0757
Fair Brier Score: 0.0752
Brier Score: 0.0627
Fair Brier Score: 0.0622
Brier Score: 0.0722
Fair Brier Score: 0.0716
Brier Score: 0.0858
Fair Brier Score: 0.0853
Brier Score: 0.0943
Fair Brier Score: 0.0940
Brier Score: 0.0583
Fair Brier Score: 0.0577
Brier Score: 0.0825
Fair Brier Score: 0.0819


/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py:1118: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  forecast_obs_df["squared_diff"] = squared_diffs
/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py:1119: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  forecast_obs_df["fair_brier_component"] = fair_brier_components
/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilis

Brier Score: 0.1140
Fair Brier Score: 0.1126
Brier Score: 0.0937
Fair Brier Score: 0.0925
Brier Score: 0.0989
Fair Brier Score: 0.0977
Brier Score: 0.0698
Fair Brier Score: 0.0688
Brier Score: 0.0660
Fair Brier Score: 0.0648
Brier Score: 0.0765
Fair Brier Score: 0.0754
Brier Score: 0.0931
Fair Brier Score: 0.0919
Brier Score: 0.0715
Fair Brier Score: 0.0704


...

Brier Score: 0.0635
Fair Brier Score: 0.0580
Brier Score: 0.0727
Fair Brier Score: 0.0675
Brier Score: 0.0803
Fair Brier Score: 0.0753
Brier Score: 0.0761
Fair Brier Score: 0.0721
Brier Score: 0.0706
Fair Brier Score: 0.0649
Brier Score: 0.0785
Fair Brier Score: 0.0731
Brier Score: 0.0976
Fair Brier Score: 0.0965
Brier Score: 0.0895
Fair Brier Score: 0.0885
Brier Score: 0.0942
Fair Brier Score: 0.0933
Brier Score: 0.0760
Fair Brier Score: 0.0749
Brier Score: 0.0713
Fair Brier Score: 0.0703
Brier Score: 0.0779
Fair Brier Score: 0.0770


/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py:1118: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  forecast_obs_df["squared_diff"] = squared_diffs
/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py:1119: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  forecast_obs_df["fair_brier_component"] = fair_brier_components
/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilis

Brier Score: 0.0861
Fair Brier Score: 0.0852
Brier Score: 0.0960
Fair Brier Score: 0.0952
Brier Score: 0.0660
Fair Brier Score: 0.0649
Brier Score: 0.0905
Fair Brier Score: 0.0895
Brier Score: 0.1087
Fair Brier Score: 0.1072
Brier Score: 0.0920
Fair Brier Score: 0.0905
Brier Score: 0.1005
Fair Brier Score: 0.0992
Brier Score: 0.0763
Fair Brier Score: 0.0749
Brier Score: 0.0721
Fair Brier Score: 0.0707
Brier Score: 0.0814
Fair Brier Score: 0.0801
Brier Score: 0.0909
Fair Brier Score: 0.0895
Brier Score: 0.0829
Fair Brier Score: 0.0816
Brier Score: 0.0779
Fair Brier Score: 0.0764
Brier Score: 0.0793
Fair Brier Score: 0.0779
Brier Score: 0.1029
Fair Brier Score: 0.0960
Brier Score: 0.0871
Fair Brier Score: 0.0806
Brier Score: 0.0843
Fair Brier Score: 0.0780
Brier Score: 0.0807
Fair Brier Score: 0.0752
Brier Score: 0.0761
Fair Brier Score: 0.0698
Brier Score: 0.0794
Fair Brier Score: 0.0734
Brier Score: 0.0831
Fair Brier Score: 0.0771
Brier Score: 0.0950
Fair Brier Score: 0.0902
Brier Scor

/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py:1118: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  forecast_obs_df["squared_diff"] = squared_diffs
/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py:1119: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  forecast_obs_df["fair_brier_component"] = fair_brier_components
/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilis

RPS: 0.3368
Fair RPS: 0.3352
Number of forecasts: 123
RPS: 0.2143
Fair RPS: 0.2130
Number of forecasts: 122
RPS: 0.2616
Fair RPS: 0.2603
Number of forecasts: 121
RPS: 0.2599
Fair RPS: 0.2587
Number of forecasts: 148
RPS: 0.2003
Fair RPS: 0.1990
Number of forecasts: 141
RPS: 0.2443
Fair RPS: 0.2429
Number of forecasts: 142
RPS: 0.2711
Fair RPS: 0.2700
Number of forecasts: 132
RPS: 0.3884
Fair RPS: 0.3876
Number of forecasts: 143
RPS: 0.1459
Fair RPS: 0.1443
Number of forecasts: 149
RPS: 0.2218
Fair RPS: 0.2202
Number of forecasts: 138
RPS: 0.4333
Fair RPS: 0.4295
Number of forecasts: 235
RPS: 0.2612
Fair RPS: 0.2578
Number of forecasts: 236
RPS: 0.3121
Fair RPS: 0.3092
Number of forecasts: 233
RPS: 0.2143
Fair RPS: 0.2112
Number of forecasts: 285
RPS: 0.2039
Fair RPS: 0.2007
Number of forecasts: 272
RPS: 0.2584
Fair RPS: 0.2553
Number of forecasts: 272
RPS: 0.3099
Fair RPS: 0.3069
Number of forecasts: 249
RPS: 0.2107
Fair RPS: 0.2072
Number of forecasts: 274
RPS: 0.2285
Fair RPS: 0.2242

In [71]:
# brier climatology
climatology_obs_df = c.multi_year_climatological_forecast_obs_pairs(
            clim_onset,
            config["common_period"],
            config["day_bins_30"],
            config["mem_num"],
            model_paths[model_name],
            date_filter_year=2022,
            file_pattern=config["file_pattern"],
            max_forecast_day=30,
            mok=config["mok"],
        )

Using 4-degree CMZ polygon coordinates

Processing target year 2004
Creating climatological forecasts for target year 2004
Using 124 years as ensemble members: [np.int64(1901), np.int64(1902), np.int64(1903), np.int64(1904), np.int64(1905), np.int64(1906), np.int64(1907), np.int64(1908), np.int64(1909), np.int64(1910), np.int64(1911), np.int64(1912), np.int64(1913), np.int64(1914), np.int64(1915), np.int64(1916), np.int64(1917), np.int64(1918), np.int64(1919), np.int64(1920), np.int64(1921), np.int64(1922), np.int64(1923), np.int64(1924), np.int64(1925), np.int64(1926), np.int64(1927), np.int64(1928), np.int64(1929), np.int64(1930), np.int64(1931), np.int64(1932), np.int64(1933), np.int64(1934), np.int64(1935), np.int64(1936), np.int64(1937), np.int64(1938), np.int64(1939), np.int64(1940), np.int64(1941), np.int64(1942), np.int64(1943), np.int64(1944), np.int64(1945), np.int64(1946), np.int64(1947), np.int64(1948), np.int64(1949), np.int64(1950), np.int64(1951), np.int64(1952), np.int6

In [78]:
test = forecast_dfs_15["IFS"]
test = test.loc[
(test["lat"] == 20.0) & (test["lon"] == 76.0)
]

In [98]:
import warnings
import traceback

warnings.filterwarnings('error', category=RuntimeWarning)

ifs_test = forecast_dfs_15["IFS"]
clim_brier = c.calculate_brier_score_climatology(climatology_obs_df)
clim_rps = p.calculate_rps(climatology_obs_df)
rows= []
for lat in ifs_test.lat.unique():
    for lon in ifs_test.lon.unique():
        print("="*50)
        print(f"Calculating for {lat}, {lon} pair")
        print("="*50)
        row = {}
        loop_df = ifs_test.loc[
            (ifs_test["lat"] == lat) & (ifs_test["lon"] == lon)
            ].copy()

        if loop_df.empty:
            print(f"Skipping {lat}, {lon} — no data")
            continue
        else:
            loop_brier = p.calculate_brier_score(loop_df)
            loop_rps = p.calculate_rps(loop_df)
            skill_scores = p.calculate_skill_scores(
                brier_forecast=loop_brier,
                rps_forecast=loop_rps,
                brier_climatology=clim_brier,
                rps_climatology=clim_rps
            )
            row["BSS"] = skill_scores["fair_brier_skill_score"]
            row["RPS"] = skill_scores["fair_rps_skill_score"]
            row["lat"] = lat
            row["lon"] = lon
            rows.append(row)

# pd.DataFrame(rows)

Calculating Brier Score excluding 'Before initialization' bin
Original samples: 10872, After filtering: 9513
Brier Score (excluding 'Before initialization'): 0.0852
Fair Brier Score (excluding 'Before initialization'): 0.0846
Bins included in calculation: ['After day 30', 'Days 1-5', 'Days 11-15', 'Days 16-20', 'Days 21-25', 'Days 26-30', 'Days 6-10']
RPS: 0.7375
Fair RPS: 0.7327
Number of forecasts: 1359
Calculating for 20.0, 76.0 pair
Brier Score: 0.1089
Fair Brier Score: 0.1024
RPS: 0.3972
Fair RPS: 0.3766
Number of forecasts: 123
SKILL SCORE CALCULATIONS
Fair Brier Skill Score (1-15 day): -0.2102
Fair RPS Skill Score (1-15 day): 0.4859

Automatically detected target bins: ['Days 1-5', 'Days 6-10', 'Days 11-15']
Excluded bins: []

Bin-wise Fair Brier Skill Scores:
  Days 1-5: Fair BSS = -0.2846
  Days 6-10: Fair BSS = 0.1171
  Days 11-15: Fair BSS = -0.1772

SKILL SCORE SUMMARY TABLE
Metric                         Overall (1-15 day) 1-5          6-10         11-15       
-----------

In [79]:
test

,init_time,lat,lon,bin_start,bin_end,bin_label,predicted_prob,observed_onset,members_with_onset,total_members,year,obs_onset_date,bin_index,squared_diff,fair_brier_component
0,2004-05-02,20.0,76.0,1,5.0,Days 1-5,0.000000,0,0,11,2004,2004-06-08,0,0.000000,0.000000
1,2004-05-02,20.0,76.0,6,10.0,Days 6-10,0.000000,0,0,11,2004,2004-06-08,1,0.000000,0.000000
2,2004-05-02,20.0,76.0,11,15.0,Days 11-15,0.000000,0,0,11,2004,2004-06-08,2,0.000000,0.000000
3,2004-05-02,20.0,76.0,16,inf,After day 15,1.000000,1,11,11,2004,2004-06-08,3,0.000000,0.000000
40,2004-05-09,20.0,76.0,1,5.0,Days 1-5,0.000000,0,0,11,2004,2004-06-08,0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5303,2021-05-23,20.0,76.0,16,inf,After day 15,0.727273,0,8,11,2021,2021-06-05,3,0.528926,0.509091
5340,2021-05-30,20.0,76.0,1,5.0,Days 1-5,0.909091,0,10,11,2021,2021-06-05,0,0.826446,0.818182
5341,2021-05-30,20.0,76.0,6,10.0,Days 6-10,0.000000,1,0,11,2021,2021-06-05,1,1.000000,1.000000
5342,2021-05-30,20.0,76.0,11,15.0,Days 11-15,0.090909,0,1,11,2021,2021-06-05,2,0.008264,0.000000


In [80]:
ifs_rps = p.calculate_rps(test)
ifs_brier = p.calculate_brier_score(test)

p.calculate_skill_scores(
    brier_forecast=ifs_brier,
    rps_forecast=ifs_rps,
    brier_climatology=clim_brier,
    rps_climatology=clim_rps
)

RPS: 0.3972
Fair RPS: 0.3766
Number of forecasts: 123
Brier Score: 0.1089
Fair Brier Score: 0.1024
Calculating Brier Score excluding 'Before initialization' bin
Original samples: 10872, After filtering: 9513
Brier Score (excluding 'Before initialization'): 0.0852
Fair Brier Score (excluding 'Before initialization'): 0.0846
Bins included in calculation: ['After day 30', 'Days 1-5', 'Days 11-15', 'Days 16-20', 'Days 21-25', 'Days 26-30', 'Days 6-10']


/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py:1118: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  forecast_obs_df["squared_diff"] = squared_diffs
/Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/monsoonbench/metrics/probabilistic.py:1119: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  forecast_obs_df["fair_brier_component"] = fair_brier_components


RPS: 0.7375
Fair RPS: 0.7327
Number of forecasts: 1359
SKILL SCORE CALCULATIONS
Fair Brier Skill Score (1-15 day): -0.2102
Fair RPS Skill Score (1-15 day): 0.4859

Automatically detected target bins: ['Days 1-5', 'Days 6-10', 'Days 11-15']
Excluded bins: []

Bin-wise Fair Brier Skill Scores:
  Days 1-5: Fair BSS = -0.2846
  Days 6-10: Fair BSS = 0.1171
  Days 11-15: Fair BSS = -0.1772

SKILL SCORE SUMMARY TABLE
Metric                         Overall (1-15 day) 1-5          6-10         11-15       
------------------------------------------------------------------------------------
Fair Brier Skill Score         -0.2102            -0.2846      0.1171       -0.1772     
Fair RPS Skill Score           0.4859             N/A          N/A          N/A         
------------------------------------------------------------------------------------

Interpretation Guide:
• Positive skill scores indicate forecast is better than climatology
• Negative skill scores indicate forecast is worse than 

{'fair_brier_skill_score': np.float64(-0.2101897945370823),
 'fair_rps_skill_score': np.float64(0.4859196760700001),
 'bin_fair_brier_skill_scores': {'Days 1-5': -0.2845795673324858,
  'Days 6-10': 0.11705038940192203,
  'Days 11-15': -0.17722929869020065}}

In [ ]:
# Climatology brier scores

### Fig. 7 - 12 -- Spatial Metrics 4x4 lon/lat grid

In [34]:
model_years = {
    "FuXi S2S": [2019, 2020, 2021],
    "IFS": [2019, 2020, 2021, 2022, 2023],
    "Standard": [2019, 2020, 2021, 2022, 2023, 2024],
}

In [37]:
# 15 day
metrics_df_clim_15, onset_da_clim_15 = (
        c.compute_climatology_baseline_multiple_years(
            years=model_years["Standard"],
            imd_folder=config["imd_folder"],
            thres_file=config["thresh_file"],
            tolerance_days=3,
            verification_window=1,
            forecast_days=15,
            max_forecast_day=15,
            mok=True,
            onset_window=5,
            mok_month=6,
            mok_day=2,
        )
    )

spatial_clim_15_day = c.create_spatial_far_mr_mae(
        metrics_df_clim_15, dict.fromkeys(model_years["Standard"], onset_da_clim_15)
    )

model_dfs_15 = {}
model_onsets_15 = {}

for model_name, model_fp in prob_model_paths.items():
    print("=" * 80)
    print(f"Loading data from {model_name}")
    print("=" * 80)
    probabilistic_df_15, onset_da_dict_15 = (
        p.compute_metrics_multiple_years(
            years=(
                model_years[model_name]
                if model_name in model_years.keys()
                else model_years["Standard"]
            ),
            imd_folder=config["imd_folder"],
            thres_file=config["thresh_file"],
            model_forecast_dir=model_fp,
            tolerance_days=3,
            verification_window=1,
            forecast_days=15,
            max_forecast_day=15,
            mok=True,
            onset_window=5,
            mok_month=6,
            mok_day=2,
        )
    )

    model_dfs_15[model_name] = probabilistic_df_15
    model_onsets_15[model_name] = onset_da_dict_15

for model_name, model_fp in det_model_paths.items():
        print("=" * 80)
        print(f"Loading data from {model_name}")
        print("=" * 80)
        deterministic_df_15, onset_da_dict_15 = (
            d.compute_metrics_multiple_years(
                years=(
                    model_years[model_name]
                    if model_name in model_years.keys()
                    else model_years["Standard"]
                ),
                imd_folder=config["imd_folder"],
                thres_file=config["thresh_file"],
                model_forecast_dir=model_fp,
                tolerance_days=3,
                verification_window=1,
                forecast_days=15,
                max_forecast_day=15,
                mok=True,
                onset_window=5,
                mok_month=6,
                mok_day=2,
            )
        )

        model_dfs_15[model_name] = deterministic_df_15
        model_onsets_15[model_name] = onset_da_dict_15

Computing climatological onset reference...
Computing climatological onset from 124 years: 1901-2024
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/1901.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1901-06-02) as start date for onset detection
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/1902.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1902-06-02) as start date for onset detection
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/1903.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1903-06-02) as start date for onset detection
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_da

In [38]:
# 30 Day
metrics_df_clim_30, onset_da_clim_30 = (
        c.compute_climatology_baseline_multiple_years(
            years=model_years["Standard"],
            imd_folder=config["imd_folder"],
            thres_file=config["thresh_file"],
            tolerance_days=5,
            verification_window=16,
            forecast_days=30,
            max_forecast_day=30,
            mok=True,
            onset_window=5,
            mok_month=6,
            mok_day=2,
        )
    )

spatial_clim_30_day = c.create_spatial_far_mr_mae(
        metrics_df_clim_30, dict.fromkeys(model_years["Standard"], onset_da_clim_30)
    )

model_dfs_30 = {}
model_onsets_30 = {}

for model_name, model_fp in prob_model_paths.items():
    print("=" * 80)
    print(f"Loading data from {model_name}")
    print("=" * 80)
    probabilistic_df_30, onset_da_dict_30 = (
        p.compute_metrics_multiple_years(
            years=(
                model_years[model_name]
                if model_name in model_years.keys()
                else model_years["Standard"]
            ),
            imd_folder=config["imd_folder"],
            thres_file=config["thresh_file"],
            model_forecast_dir=model_fp,
            tolerance_days=5,
            verification_window=16,
            forecast_days=30,
            max_forecast_day=30,
            mok=True,
            onset_window=5,
            mok_month=6,
            mok_day=2,
        )
    )
    model_dfs_30[model_name] = probabilistic_df_30
    model_onsets_30[model_name] = onset_da_dict_30

for model_name, model_fp in det_model_paths.items():
    print("=" * 80)
    print(f"Loading data from {model_name}")
    print("=" * 80)
    deterministic_df_30, onset_da_dict_30 = (
        d.compute_metrics_multiple_years(
            years=(
                model_years[model_name]
                if model_name in model_years.keys()
                else model_years["Standard"]
            ),
            imd_folder=config["imd_folder"],
            thres_file=config["thresh_file"],
            model_forecast_dir=model_fp,
            tolerance_days=5,
            verification_window=16,
            forecast_days=30,
            max_forecast_day=30,
            mok=True,
            onset_window=5,
            mok_month=6,
            mok_day=2,
        )
    )

    model_dfs_30[model_name] = deterministic_df_30
    model_onsets_30[model_name] = onset_da_dict_30


Computing climatological onset reference...
Computing climatological onset from 124 years: 1901-2024
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/1901.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1901-06-02) as start date for onset detection
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/1902.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1902-06-02) as start date for onset detection
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/1903.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1903-06-02) as start date for onset detection
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_da